### Exception hierarchy


In Python exceptions are objects, like all values in Python. These objects are instantiated from exception classes. Exception classes form natural hierarchies:

* New exception classes can be made by inheriting from existing exception classes and extending them
* The root of this hierarchy is the class Exception
* Python defines several base classes to derive from, and several ready-to-use exception classes

<img src="images/exception_hierarchy.png" width="500"/>

### Too general exception specifications


The exception hierarchy allows to catch multiple similar exceptions by catching their common base class. This feature has to be used carefully. Over-general exception specification, like except ```Exception:```, can hide the real reason for an error. Example of this:

In [6]:
import sys

s=input('Give a number: ')
s=s[:-1] #String the /n character from the end

try:
    x=int(s)
    totttttol +=x
    sys.stdout.write(f'You entered {x}\n')
except Exception:
    print('You did not enter a number.')

You did not enter a number.


In the previous example, if the user doesn’t enter a string that represents an integer, a ```ValueError``` is raised by the ```int``` function.

Instead of catching the ```ValueError```, we catch the root of the exception hierarchy, namely ```Exception```. This results in catching all possible exceptions. But this will cause one typing error in the program to go undetected. Change the exception specification from Exception to ValueError to see what this error is.

## "One Typing Error in the Program to Go Undetected" — Simply

This is warning about a **real danger** of catching `Exception` too broadly — and it's making a specific claim: **there's an actual bug (a typo) hiding somewhere in this example's code**, and catching `Exception` is exactly what's **hiding** it from you.

---

### The Core Problem — `except Exception:` Catches EVERYTHING

Remember `Exception` sits at the **root** of almost the entire exception family tree — every specific exception type (`ValueError`, `TypeError`, `ZeroDivisionError`, `NameError`, `AttributeError`...) is a **descendant** of it:

```
Exception
    ├── ValueError
    ├── TypeError
    ├── ZeroDivisionError
    ├── NameError
    ├── AttributeError
    └── ... (nearly everything)
```

```python
except Exception:
    ...
```

This says: *"catch ANY of these — I don't care which one."* And that's precisely the danger.

---

### The Specific Bug Being Described — A Typo, Disguised as an Expected Error

Imagine the example code looks something like this (a typical "keep asking until valid input" loop):

```python
while True:
    try:
        x = int(input("Enter a number: "))
        break
    except Exception:              # ← catches EVERYTHING
        print("Please enter a valid integer")
```

Now suppose there's an **accidental typo** somewhere in the real code — say, the variable was misspelled:

```python
while True:
    try:
        x = int(input("Enter a number: "))
        totall += x          # ← TYPO! Should be 'total', not 'totall'
        break
    except Exception:
        print("Please enter a valid integer")
```

`totall` doesn't exist → this raises `NameError: name 'totall' is not defined`. But because `except Exception:` catches **absolutely everything**, including `NameError` — **the typo gets silently swallowed** and the program just prints the misleading message:

```
Please enter a valid integer
```

**Even if you typed a perfectly valid integer!** The message is a **lie** — the real problem was a typo in the programmer's own code, but it got **disguised** as "you entered bad input," because both errors funnel into the same overly-broad `except Exception:` block.

---

### Why This Is Genuinely Dangerous

You, the programmer, would be staring at correct input being rejected, with **zero clue** that the actual bug is a misspelled variable name three lines away — because the exception handler **hid the real error message** from you entirely.

```python
except Exception:
    print("Please enter a valid integer")     # generic message — masks WHICH exception actually fired!
```

Compare with narrowly catching only `ValueError`:

```python
while True:
    try:
        x = int(input("Enter a number: "))
        totall += x            # ← the SAME typo
        break
    except ValueError:            # ← ONLY catches ValueError now
        print("Please enter a valid integer")
```

Now, when `NameError` fires (from `totall`), **there's no matching `except` for it** — the program **crashes with the real error**, showing you the actual traceback:

```
NameError: name 'totall' is not defined
```

**That's the fix the exercise is asking you to try** — narrowing `Exception` down to `ValueError` specifically **reveals** the hidden typo bug, because `NameError` is no longer being silently absorbed by an overly broad `except`.

---

### The General Lesson — Broad `except` Clauses Hide Bugs

```python
try:
    risky_code()
except Exception:          # ✗ "catch anything" — masks unrelated bugs
    handle_it()
```

```python
try:
    risky_code()
except ValueError:          # ✓ catches only the SPECIFIC error you anticipated
    handle_it()
```

**Rule of thumb:** always catch the **narrowest, most specific** exception type that you actually expect and know how to handle. A broad `except Exception:` (or worse, a bare `except:`) is a trap — it will happily swallow **any** bug, expected or not, and give you no way to tell the difference between "the user typed garbage" and "I have a real bug in my code."

---

### Connecting to Your Earlier Regex Lesson

This is the **exact same principle** as when you correctly caught `ValueError` specifically (not a bare `except:`) in your `summary()` function's number-parsing! At the time, the reasoning was: *"a bare `except:` would also silently swallow unrelated bugs, making them invisible."* This exercise is now **demonstrating** that exact warning with a concrete, hands-on example — showing you a typo that genuinely disappears behind `except Exception:`, and reappears the moment you narrow it to `except ValueError:`.

---

### The One-Sentence Summary

> Catching the broad `Exception` class means **any** error — including a genuine programming mistake like a misspelled variable name (a `NameError`) — gets silently absorbed by the same handler meant for expected user-input mistakes, disguising a real bug as if it were just "invalid input." Narrowing the `except` to specifically `ValueError` lets that hidden typo's real error **surface and crash visibly**, which is exactly what the exercise wants you to discover by making the change yourself. 🎯

In [5]:
import sys

s=input('Give a number: ')
s=s[:-1] #String the /n character from the end

try:
    x=int(s)
    tottal +=x
    sys.stdout.write(f'You entered {x}\n')
except ValueError:
    print('You did not enter a number.')

NameError: name 'tottal' is not defined

## You're Making a Really Important Point — Let's Confirm It's Correct

Yes — this is a genuine, well-known difference between Python and Java, and your observation is accurate. Let's make it concrete.

---

### Java Would Catch This BEFORE Running Anything

```java
int x = Integer.parseInt(scanner.nextLine());
totall += x;      // ← 'totall' was never declared!
```

Java is **statically typed and compiled** — it checks that every variable name exists, matches expected types, etc., **before** the program ever runs, during the **compilation** step:

```
javac Program.java

Program.java:5: error: cannot find symbol
        totall += x;
        ^
  symbol:   variable totall
```

**The program never even starts.** You're forced to fix the typo first — there's no way to accidentally ship or run code with this mistake still in it.

---

### Python Only Discovers This WHILE Running

Python is **dynamically typed and interpreted** — there's no separate "compile the whole program and check everything" step. Python just starts executing your code, line by line, and only discovers `totall` doesn't exist **at the exact moment** that line actually runs:

```python
while True:
    try:
        x = int(input("Enter a number: "))
        totall += x          # NOT checked until THIS LINE actually executes
        break
    except ValueError:
        print("Please enter a valid integer")
```

If the user **never** manages to type a valid integer (they keep making typos, or you never reach this branch during testing), **this line might never run at all** — and the typo stays **completely invisible**, potentially for a long time.

---

### Why This Makes Your Earlier Concern Even More Real

Combine this with the `except Exception:` problem from before, and you get a genuinely nasty combination:

```python
try:
    x = int(input("Enter a number: "))
    totall += x
    break
except Exception:              # catches EVERYTHING, including NameError
    print("Please enter a valid integer")
```

Here's the realistic, worrying sequence:
1. You write this code with the typo, run it, type a valid number like `5`
2. `totall += x` raises `NameError` (typo)
3. `except Exception:` **silently swallows it**
4. You see `"Please enter a valid integer"` — even though you typed `5` correctly!
5. **You might conclude your input-validation logic is broken**, and go debug the wrong thing entirely — while the real bug (a misspelled variable) sits completely hidden

This is a genuinely realistic and frustrating debugging trap **specific to dynamically-typed, broad-exception-catching code** — and it simply **cannot happen** in Java, because the compiler refuses to build the program at all until `totall` is either declared or corrected.

---

### The Trade-off — Why Python Still Does It This Way

This isn't a "Python is worse" situation — it's a genuine **trade-off**:

| | Java (static, compiled) | Python (dynamic, interpreted) |
|---|---|---|
| Typos like `totall` | caught at **compile time**, before running | only discovered **at runtime**, if that line executes |
| Development speed | slower to iterate — must compile each time | faster to iterate — run instantly, no compile step |
| Safety net | strong — whole program verified upfront | weaker — only what actually executes gets checked |
| Flexibility | rigid — every type/name must be declared | flexible — variables can appear dynamically |

Python's speed and flexibility (rapid prototyping, no compile step, dynamic typing) are genuinely valuable — but they come at the **direct cost** of exactly the safety you're describing. This is one of the most commonly cited trade-offs between the two language philosophies.

---

### Why This Directly Explains the Exercise's Warning

This is **precisely** why the exercise emphasizes narrowing `except Exception:` down to `except ValueError:` specifically. Since Python has **no compiler** to catch `totall` for you ahead of time, **your exception handling is the ONLY safety net left** — and if that net is too broad (`except Exception:`), it actively **defeats itself**, hiding the one signal (`NameError` at runtime) that would have told you something was wrong.

```
Java:    compiler catches the typo    →  program won't even start
Python:  NO compiler safety net        →  ONLY chance to catch it is
                                            a correctly-scoped except clause
                                            (or the bug goes unnoticed indefinitely)
```

---

### The One-Sentence Summary

> You're completely correct — Java's compiler would refuse to build this program at all until `totall` is fixed, because it verifies every variable reference **before** execution begins. Python has no such upfront check; it only discovers `totall` doesn't exist **the moment that exact line actually runs** — which means a broad `except Exception:` is especially dangerous in Python specifically, since it's often the **only** thing standing between you and silently shipping a typo that a compiled language would have caught for free. 🎯